# Econ 148 — Project 3: GDP Nowcasting from Mixed-Frequency Indicators
**Track A | Spring 2026 | UC Berkeley**

We nowcast current-quarter US real GDP growth using monthly FRED indicators released before the BEA advance estimate. We compare an OLS baseline against Ridge/LASSO and gradient boosting, benchmarked against Atlanta Fed GDPNow.

## 0. Setup
Install required packages and set the FRED API key.

In [ ]:
!pip install fredapi pandas matplotlib seaborn scikit-learn -q

Set your FRED API key below. Get a free key at https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
from fredapi import Fred
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FRED_API_KEY = 'YOUR_API_KEY_HERE'
START_DATE = '2000-01-01'
END_DATE = '2024-12-31'

fred = Fred(api_key=FRED_API_KEY)

## 1. Data Collection
Pull GDP (quarterly) and all monthly indicators from FRED. Each series is stored in a dictionary keyed by its human-readable name.

In [ ]:
SERIES = {
    'GDP': 'GDP',
    'Payrolls': 'PAYEMS',
    'Industrial_Production': 'INDPRO',
    'Retail_Sales': 'RSAFS',
    'Consumer_Sentiment': 'UMCSENT',
    'Unemployment': 'UNRATE',
    'Treasury_10Y': 'GS10',
    'CPI': 'CPIAUCSL',
}

raw = {}
for name, sid in SERIES.items():
    raw[name] = fred.get_series(sid, START_DATE, END_DATE)
    print(f'Pulled {name}: {len(raw[name])} obs')

## 2. Mixed-Frequency Alignment
GDP is quarterly; indicators are monthly. We resample monthly series to quarterly frequency using end-of-quarter values, then join on a common quarterly index.

In [ ]:
gdp = raw.pop('GDP').dropna()
gdp.index = pd.to_datetime(gdp.index)

monthly = pd.DataFrame(raw)
monthly.index = pd.to_datetime(monthly.index)

quarterly = monthly.resample('QS').last()
df = quarterly.join(gdp.rename('GDP'), how='inner').dropna()

print('Combined dataframe shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis
Plot each indicator against GDP to visualize co-movement. This helps us understand which series are most correlated with GDP before modeling.

In [ ]:
features = [c for c in df.columns if c != 'GDP']
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].plot(df.index, df[col], color='steelblue', label=col)
    ax2 = axes[i].twinx()
    ax2.plot(df.index, df['GDP'], color='coral', linestyle='--', alpha=0.7, label='GDP')
    axes[i].set_title(col)

plt.suptitle('Monthly Indicators vs GDP', fontsize=14)
plt.tight_layout()
plt.savefig('eda_indicators.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering
Compute year-over-year growth rates for each indicator. Growth rates are more stationary than levels and align better with GDP growth as a target variable.

In [ ]:
df_growth = df.pct_change(4).dropna()
print('Growth rate dataframe shape:', df_growth.shape)
df_growth.head()

## 5. Model Training
Placeholder section — models (OLS, Ridge, LASSO, Gradient Boosting) will be added here in subsequent work.

In [ ]:
print('Model training coming in next iteration.')